# Gold Layer - Business KPIs

## Objective

Create business-facing KPI datasets from curated Silver and dimensional tables.

### Business Questions

1. Which borough generates the highest trip demand and revenue?
2. How are ride events distributed across statuses and dates?
3. What does the current customer population look like?
4. Which boroughs generate the highest taxi activity?
5. How does trip duration change across consecutive rides?
6. What weather conditions were observed during the reporting period?

### Inputs

- workspace.default.silver_trips
- workspace.default.silver_weather
- workspace.default.silver_ride_events
- workspace.default.dim_customers_scd2

### Outputs

- workspace.default.gold_trip_kpis
- workspace.default.gold_event_kpis
- workspace.default.gold_customer_kpis
- workspace.default.gold_top_boroughs
- workspace.default.gold_trip_timeline
- workspace.default.gold_weather_kpis
- workspace.default.gold_event_status_pivot

## Read Curated Tables

In [0]:
from pyspark.sql.functions import (
    count,
    sum,
    avg,
    min,
    max,
    col,
    lead,
    lag,
    row_number,
    rank,
    dense_rank
)

from pyspark.sql.window import Window

silver_trips = spark.table(
    "workspace.default.silver_trips"
)

silver_events = spark.table(
    "workspace.default.silver_ride_events"
)

silver_weather = spark.table(
    "workspace.default.silver_weather"
)

customers = spark.table(
    "workspace.default.dim_customers_scd2"
)

In [0]:
# KPI: Measure trip demand, revenue and trip performance by pickup borough.

gold_trip_kpis = (
    silver_trips
    .groupBy("pickup_borough")
    .agg(
        count("*").alias("trip_count"),
        avg("fare_amount").alias("avg_fare"),
        avg("trip_distance").alias("avg_trip_distance"),
        avg("trip_duration_minutes").alias(
            "avg_trip_duration"
        ),
        sum("total_amount").alias(
            "total_revenue"
        ),
        max("fare_amount").alias("max_fare"),
        min("fare_amount").alias("min_fare")
    )
)

display(gold_trip_kpis)

In [0]:
# KPI: Analyze ride-event activity and completed event volume by day.

gold_event_kpis = (
    silver_events
    .groupBy(
        "ride_status",
        "event_date"
    )
    .agg(
        count("*").alias(
            "total_events"
        ),
        sum("is_completed").alias(
            "completed_event_count"
        )
    )
)

display(gold_event_kpis)

In [0]:
# KPI: Understand customer distribution and loyalty by membership tier.

gold_customer_kpis = (
    customers
    .filter(
        col("is_current")
    )
    .groupBy("membership_type")
    .agg(
        count("*").alias(
            "customer_count"
        ),
        avg("loyalty_points").alias(
            "avg_loyalty_points"
        )
    )
)

display(gold_customer_kpis)

## Window Function Analysis

The following KPIs demonstrate analytical window functions commonly used in business reporting:

- ROW_NUMBER()
- RANK()
- DENSE_RANK()
- LEAD()
- LAG()

In [0]:
# KPI: Rank boroughs by trip volume using window functions.

borough_window = Window.orderBy(
    col("trip_count").desc()
)

gold_top_boroughs = (
    gold_trip_kpis
    .withColumn(
        "row_number",
        row_number().over(
            borough_window
        )
    )
    .withColumn(
        "rank",
        rank().over(
            borough_window
        )
    )
    .withColumn(
        "dense_rank",
        dense_rank().over(
            borough_window
        )
    )
)

display(gold_top_boroughs)

In [0]:
# KPI: Compare consecutive trip durations using LEAD and LAG.

trip_window = (
    Window
    .partitionBy(
        "pickup_borough"
    )
    .orderBy(
        "tpep_pickup_datetime"
    )
)

gold_trip_timeline = (
    silver_trips
    .select(
        "tpep_pickup_datetime",
        "trip_duration_minutes",
        "fare_amount",
        "pickup_borough"
    )
    .withColumn(
        "previous_trip_duration",
        lag(
            "trip_duration_minutes"
        ).over(trip_window)
    )
    .withColumn(
        "next_trip_duration",
        lead(
            "trip_duration_minutes"
        ).over(trip_window)
    )
)

display(
    gold_trip_timeline.limit(50)
)

In [0]:
# KPI: Summarize daily weather conditions for reporting and future correlation analysis.

gold_weather_kpis = (
    silver_weather
    .groupBy(
        "weather_date"
    )
    .agg(
        avg("max_temperature").alias(
            "avg_max_temperature"
        ),
        avg("min_temperature").alias(
            "avg_min_temperature"
        ),
        sum("precipitation").alias(
            "total_precipitation"
        )
    )
)

display(gold_weather_kpis)

In [0]:
# KPI: Analyze daily ride-event distribution across ride statuses.

gold_event_status_pivot = (
    silver_events
    .groupBy("event_date")
    .pivot("ride_status")
    .count()
)

display(gold_event_status_pivot)

## Create Gold Tables

In [0]:
gold_trip_kpis.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.default.gold_trip_kpis"
    )

gold_event_kpis.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.default.gold_event_kpis"
    )

gold_customer_kpis.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.default.gold_customer_kpis"
    )

gold_top_boroughs.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.default.gold_top_boroughs"
    )

gold_trip_timeline.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.default.gold_trip_timeline"
    )

gold_weather_kpis.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.default.gold_weather_kpis"
    )

gold_event_status_pivot = gold_event_status_pivot.toDF(
    *[c.replace(" ", "_") for c in gold_event_status_pivot.columns]
)

gold_event_status_pivot.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.default.gold_event_status_pivot"
    )

## Validation

In [0]:
%sql
SELECT *
FROM workspace.default.gold_trip_kpis
ORDER BY total_revenue DESC

In [0]:
%sql
SELECT *
FROM workspace.default.gold_event_kpis

In [0]:
%sql
SELECT *
FROM workspace.default.gold_customer_kpis

In [0]:
%sql
SELECT *
FROM workspace.default.gold_top_boroughs
ORDER BY rank

In [0]:
%sql
SELECT *
FROM workspace.default.gold_trip_timeline
LIMIT 20

In [0]:
%sql
SELECT *
FROM workspace.default.gold_weather_kpis

In [0]:
%sql
SELECT *
FROM workspace.default.gold_event_status_pivot